In [ ]:
%%writefile hello.cu
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include "cuda_runtime.h"
#include "device_launch_parameters.h"

__global__ void AddVecGPU(float *c, float *a, float *b);

int main() {
  int N = 256; //N의 최대값
  //하나의 블록에는 1024개의 thread가 최대임.
  //그 이상으로 늘리려면 block과 grid 모두 늘려야됨.

  float *a = new float[N];
  float *b = new float[N];
  float *c = new float[N];
  for(int i = 0; i < N; ++i) {
    a[i] = rand() / (float)RAND_MAX;
    b[i] = -a[i];
    c[i] = 0.0;
  }

  //사용할 GPU를 설정한다. (0번쨰 GPU)
  cudaError_t cudaStatus = cudaSetDevice(0);
  if(cudaStatus != cudaSuccess) {
    printf("Error\n");
    return 1;
  }

  //GPU에 메모리를 할당
  float *dev_a, *dev_b, *dev_c;
  cudaStatus = cudaMalloc((void**)&dev_a, N * sizeof(float));
  cudaStatus = cudaMalloc((void**)&dev_b, N * sizeof(float));
  cudaStatus = cudaMalloc((void**)&dev_c, N * sizeof(float));

  //CPU 배열의 데이터를 GPU 메모리에 복사한다.
  cudaStatus = cudaMemcpy(dev_a, a, N * sizeof(float), cudaMemcpyHostToDevice);
  cudaStatus = cudaMemcpy(dev_b, b, N * sizeof(float), cudaMemcpyHostToDevice);
  cudaStatus = cudaMemcpy(dev_c, c, N * sizeof(float), cudaMemcpyHostToDevice);

  clock_t st = clock(); //시간 측정 시작
  AddVecGPU<<<1,N>>>(dev_c, dev_a, dev_b);
  cudaStatus = cudaDeviceSynchronize();
  clock_t ed = clock(); //시간 측정 끝
  printf("Elapsed time : %u ms\n", ed - st); //%u는 unsigned int

  //결과를 GPU메모리에서 CPU 메모리로 복사한다
  cudaStatus = cudaMemcpy(c, dev_c, N * sizeof(float), cudaMemcpyDeviceToHost);

  for(int i = 0; i < N; ++i) {
    if(c[i] != 0.0) printf("Error\n");
  }

  c[0]++;
  printf("CPU c value : %f\n", c[0]);

  delete[] a;
  delete[] b;
  delete[] c;
  cudaFree(dev_c);
  cudaFree(dev_b);
  cudaFree(dev_a);
  return 0;

}

__global__ void AddVecGPU(float *c, float *a, float*b) {
  int i = threadIdx.x;
  c[i] = a[i] + b[i];
  if(i == 0) printf("dev_c value : %f\n", c[i]);
}



Overwriting hello.cu


In [ ]:
!nvcc hello.cu -o hello
!./hello

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
hello.cu: In function ‘int main()’:
hello.cu:45:8: warning: format ‘%u’ expects argument of type ‘unsigned int’, but argument 2 has type ‘clock_t’ {aka ‘long int’} []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wformat=-Wformat=]8;;]
   45 |   printf("Elapsed time : %u ms\n", ed - st); //%u는 unsigned int
      |        ^~~~~~~~~~~~~~~~~~~~~~~~  ~~~~~~~
      |                                     |
      |                                     clock_t {aka long int}
dev_c value : 0.000000
Elapsed time : 2752 ms
CPU c value : 1.000000
